# Clase 09 — Ejercicio intergrador Clínica MediCare

> Notebook de práctica. Ejecutá las celdas a medida que avanzás en el artículo.

La clínica MediCare lleva varios años registrando turnos médicos en un sistema interno. El sistema fue cambiando de versión con el tiempo, y los datos se fueron exportando a una base de datos SQLite sin demasiado control de calidad.

El director médico necesita un informe con algunas preguntas concretas sobre el funcionamiento de la clínica, pero antes de poder responderlas, alguien tiene que hacer la limpieza. Ese trabajo es el de ustedes, no se les dice qué problemas hay en los datos. Parte del ejercicio es encontrarlos.

> El archivo de la base de datos `medicare.db` ya está disponible para que lo uses en este ejercicio.

### 1) Exploración: ¿qué tiene esta base de datos?

Antes de tocar nada, el primer trabajo es entender qué hay adentro. Conectense a `medicare.db` y respondan estas preguntas.

1. ¿Qué tablas tiene la base de datos?
2. ¿Qué columnas y tipos tiene cada tabla?
3. ¿Cuántas filas tiene cada tabla?

In [1]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

tablas = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
""", conexion)

print(tablas)





        name
0  pacientes
1    medicos
2     turnos


In [2]:
for tabla in ["pacientes", "medicos", "turnos"]:
    print(f"\nTabla: {tabla}")

    columnas = pd.read_sql_query(
        f"PRAGMA table_info({tabla});",
        conexion
    )

    print(columnas[["name", "type"]])


Tabla: pacientes
               name     type
0                id  INTEGER
1            nombre     TEXT
2               dni     TEXT
3  fecha_nacimiento     TEXT
4            ciudad     TEXT
5       obra_social     TEXT

Tabla: medicos
           name     type
0            id  INTEGER
1        nombre     TEXT
2  especialidad     TEXT
3     matricula     TEXT
4        activo     TEXT

Tabla: turnos
          name     type
0           id  INTEGER
1  paciente_id  INTEGER
2    medico_id  INTEGER
3        fecha     TEXT
4         hora     TEXT
5       motivo     TEXT
6        costo     TEXT
7       estado     TEXT


In [3]:
for tabla in ["pacientes", "medicos", "turnos"]:
    consulta = f"SELECT COUNT(*) AS cantidad FROM {tabla}"

    resultado = pd.read_sql_query(consulta, conexion)

    print(f"{tabla}: {resultado.loc[0, 'cantidad']} filas")

pacientes: 66 filas
medicos: 20 filas
turnos: 210 filas


In [4]:
estados = pd.read_sql_query("""
    SELECT estado, COUNT(*) AS cantidad
    FROM turnos
    GROUP BY estado;
""", conexion)

print(estados)

         estado  cantidad
0     cancelado         2
1    realizado          6
2       AUSENTE         5
3    Cancelado          3
4     Realizado         3
5       ausente        29
6     cancelado        22
7     pendiente        25
8    pendiente          5
9     realizado       105
10     realzado         5


In [5]:
estados_normalizados = pd.read_sql_query("""
    SELECT LOWER(TRIM(estado)) AS estado_normalizado,
           COUNT(*) AS cantidad
    FROM turnos
    GROUP BY LOWER(TRIM(estado))
    ORDER BY estado_normalizado;
""", conexion)

print(estados_normalizados)

  estado_normalizado  cantidad
0            ausente        34
1          cancelado        27
2          pendiente        30
3          realizado       114
4           realzado         5


**Documenten sus hallazgos.** Antes de seguir al Paso 2, escriban en comentarios qué encontraron.

## Documentación de hallazgos — Exploración inicial

### 1. Tablas identificadas

La base de datos `medicare.db` contiene tres tablas:
- `pacientes`
- `medicos`
- `turnos`

### 2. Estructura de las tablas

**Tabla `pacientes`:**
- `id` (INTEGER)
- `nombre` (TEXT)
- `dni` (TEXT)
- `fecha_nacimiento` (TEXT)
- `ciudad` (TEXT)
- `obra_social` (TEXT)

**Tabla `medicos`:**
- `id` (INTEGER)
- `nombre` (TEXT)
- `especialidad` (TEXT)
- `matricula` (TEXT)
- `activo` (TEXT)

**Tabla `turnos`:**
Se identificaron las columnas `id`, `hora`, `motivo`, `costo` y `estado`.
La salida obtenida no permitió visualizar todas las columnas, por lo que queda pendiente completar su estructura.

### 3. Cantidad de registros

| Tabla | Cantidad de filas |
|---|---:|
| pacientes | 66 |
| medicos | 20 |
| turnos | 210 |
| **Total** | **296** |

### 4. Hallazgos preliminares

- La tabla `turnos` contiene más registros que `pacientes`. Esto puede deberse a que un paciente puede tener varios turnos.
- La columna `costo` de `turnos` está definida como `TEXT`. Conviene revisar su contenido antes de realizar cálculos numéricos.
- La columna `estado` presenta diferencias de escritura: hay valores con mayúsculas y minúsculas diferentes, y aparece la variante `realzado` en 5 registros. Se debe verificar si corresponde a un error de escritura de `realizado` antes de corregirlo.

### Conclusión

La exploración permitió conocer las tablas, identificar parte de su estructura y contabilizar sus registros. También se detectaron aspectos que requieren una revisión posterior durante el diagnóstico de calidad de los datos.

### 2) Diagnóstico: ¿qué problemas tienen los datos?

Traigan las tres tablas a DataFrames y hagan un diagnóstico completo de cada una. Para cada tabla respondan:

- ¿Cuántos nulos hay por columna? ¿Qué porcentaje representan?
- ¿Hay filas duplicadas? ¿Cuáles?
- ¿Hay columnas con tipos incorrectos que van a impedir operar con los datos?
- ¿Hay valores que parecen errores aunque no sean nulos? (Pista: revisen la columna `costo` en `turnos`)


In [6]:
pacientes = pd.read_sql_query("SELECT * FROM pacientes", conexion)
medicos = pd.read_sql_query("SELECT * FROM medicos", conexion)
turnos = pd.read_sql_query("SELECT * FROM turnos", conexion)


In [7]:
for nombre, df in [
    ("pacientes", pacientes),
    ("medicos", medicos),
    ("turnos", turnos)
]:
    print(f"\nTabla: {nombre}")
    diagnostico = pd.DataFrame({
        "nulos": df.isnull().sum(),
        "porcentaje_nulos": (df.isnull().mean() * 100).round(2)
    })
    print(diagnostico)


Tabla: pacientes
                  nulos  porcentaje_nulos
id                    0              0.00
nombre                0              0.00
dni                   7             10.61
fecha_nacimiento     11             16.67
ciudad                8             12.12
obra_social          12             18.18

Tabla: medicos
              nulos  porcentaje_nulos
id                0               0.0
nombre            0               0.0
especialidad      0               0.0
matricula         2              10.0
activo            4              20.0

Tabla: turnos
             nulos  porcentaje_nulos
id               0              0.00
paciente_id      0              0.00
medico_id        0              0.00
fecha            0              0.00
hora            16              7.62
motivo          16              7.62
costo            9              4.29
estado           0              0.00


**Antes de seguir al Paso 3, escriban una lista con todos los problemas que encontraron, uno por línea.** No empiecen a limpiar todavía.

## Diagnóstico inicial: problemas encontrados

- En la tabla `pacientes`, la columna `dni` tiene 7 valores nulos (10,61 %).
- En la tabla `pacientes`, la columna `fecha_nacimiento` tiene 11 valores nulos (16,67 %).
- En la tabla `pacientes`, la columna `ciudad` tiene 8 valores nulos (12,12 %).
- En la tabla `pacientes`, la columna `obra_social` tiene 12 valores nulos (18,18 %).
- En la tabla `medicos`, la columna `matricula` tiene 2 valores nulos (10 %).
- En la tabla `medicos`, la columna `activo` tiene 4 valores nulos (20 %).
- En la tabla `turnos`, la columna `hora` tiene 16 valores nulos (7,62 %).
- En la tabla `turnos`, la columna `motivo` tiene 16 valores nulos (7,62 %).
- En la tabla `turnos`, la columna `costo` tiene 9 valores nulos (4,29 %).
- La columna `costo` de `turnos` está definida como `TEXT`, por lo que es necesario revisar si sus valores permiten realizar operaciones numéricas.
- La columna `estado` de `turnos` presenta diferencias de escritura y mayúsculas/minúsculas; además, se encontraron 5 registros con el valor `realzado`, que podría ser un error de escritura de `realizado`. Debe verificarse antes de corregirlo.

**Nota:** Los duplicados y otros posibles errores se agregarán a esta lista después de completar las comprobaciones correspondientes.

### 3) Decisiones: ¿qué hacemos con cada problema?

Este es el paso más importante del ejercicio. Para cada problema que encontraron en el Paso 2, decidan qué van a hacer y por qué. No hay una respuesta única correcta — lo que importa es que la decisión sea coherente con el análisis que necesitan hacer después.

Completen esta tabla antes de escribir código:

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| pacientes | ciudad | 2 nulos | ... | ... |
| pacientes | dni | 1 nulo | ... | ... |
| pacientes | ciudad | 8 nulos | Imputar con "Desconocida" | Evita perder registros al agrupar geográficamente. |
| pacientes | dni | 7 nulos | Rellenar con "Sin DNI" | El DNI no es crítico para las métricas principales. |
| pacientes | fecha_nacimiento | 11 nulos | Dejar como nulos | Las preguntas no requieren calcular edades. |
| pacientes | obra_social | 12 nulos | Reemplazar por "Sin obra social" | Permite contabilizar pacientes sin cobertura directamente. |
| medicos | matricula | 2 nulos | Rellenar con "Sin matrícula" | No afecta el conteo de turnos ni la facturación. |
| medicos | activo | 4 nulos | Completar con "Desconocido" | El análisis es histórico y no invalida turnos pasados. |
| turnos | hora | 16 nulos | Dejar como nulos | No interfieren en el cálculo de costos ni estados. |
| turnos | motivo | 16 nulos | Dejar como nulos | No se requieren para responder las preguntas analíticas. |
| turnos | costo | 9 nulos y tipo TEXT | Rellenar con 0 y convertir a float | Necesario para poder sumar la facturación por especialidad. |
| turnos | estado | Inconsistencias y 5 "realzado" | Normalizar minúsculas, TRIM y corregir a "realzado" $\to$ "realizado" | Evita subcontabilizar turnos por errores de tipeo. |


### 4) Limpieza: aplicar las decisiones

Ahora sí, implementen las decisiones del Paso 3. Recuerden:

- Primero eliminen duplicados
- Después manejen nulos
- Después conviertan tipos

In [8]:
# Limpieza de pacientes
df_pacientes_limpio = pacientes.copy()
df_pacientes_limpio = df_pacientes_limpio.drop_duplicates()
df_pacientes_limpio['ciudad'] = df_pacientes_limpio['ciudad'].fillna('Desconocida')
df_pacientes_limpio['dni'] = df_pacientes_limpio['dni'].fillna('Sin DNI')
df_pacientes_limpio['obra_social'] = df_pacientes_limpio['obra_social'].fillna('Sin obra social')

# Limpieza de médicos
df_medicos_limpio = medicos.copy()
df_medicos_limpio = df_medicos_limpio.drop_duplicates()
df_medicos_limpio['matricula'] = df_medicos_limpio['matricula'].fillna('Sin matrícula')
df_medicos_limpio['activo'] = df_medicos_limpio['activo'].fillna('Desconocido')


# Limpieza de turnos
df_turnos_limpio = turnos.copy()
df_turnos_limpio = df_turnos_limpio.drop_duplicates()
df_turnos_limpio['costo'] = df_turnos_limpio['costo'].fillna(0)
df_turnos_limpio['costo'] = (
    df_turnos_limpio['costo']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df_turnos_limpio['costo'] = pd.to_numeric(df_turnos_limpio['costo'], errors='coerce').fillna(0)
df_turnos_limpio['estado'] = (
    df_turnos_limpio['estado']
    .str.lower()
    .str.strip()
    .replace('realzado', 'realizado')
)

> **Por qué `.copy()`:** cuando hacen `df_pacientes_limpio = df_pacientes`, no están creando una copia — están apuntando al mismo objeto. Cualquier cambio en `df_pacientes_limpio` modifica también `df_pacientes`. `.copy()` crea una copia independiente, así conservan los datos originales para comparar.

Al terminar, impriman un resumen de qué cambió en cada tabla:

In [9]:
print("--- RESUMEN DE CAMBIOS ---")
print(f"Pacientes limpios: {len(df_pacientes_limpio)} registros (Duplicados removidos)")
print(f"Médicos limpios: {len(df_medicos_limpio)} registros")
print(f"Turnos limpios: {len(df_turnos_limpio)} registros")
print(f"Nulos en costo de turnos: {df_turnos_limpio['costo'].isnull().sum()}")
print(f"Estados únicos normalizados: {df_turnos_limpio['estado'].unique()}")

--- RESUMEN DE CAMBIOS ---
Pacientes limpios: 66 registros (Duplicados removidos)
Médicos limpios: 20 registros
Turnos limpios: 210 registros
Nulos en costo de turnos: 0
Estados únicos normalizados: <StringArray>
['realizado', 'cancelado', 'ausente', 'pendiente']
Length: 4, dtype: str


### 5) — Análisis: responder las preguntas del director

Con los datos limpios, respondan estas cuatro preguntas. Para cada una, combinen SQL (para traer los datos necesarios) con Pandas (para calcular o presentar el resultado):

**Pregunta 1:** ¿Cuántos turnos realizó cada médico? Mostrá el resultado ordenado de mayor a menor.

**Pregunta 2:** ¿Cuál es el costo total facturado por especialidad médica? Considerá solo los turnos con estado `'realizado'`.

**Pregunta 3:** ¿Qué obra social tiene más pacientes registrados? ¿Y cuántos pacientes no tienen obra social?

**Pregunta 4:** ¿Cuántos turnos fueron cancelados o marcados como `'ausente'`? ¿Qué médico tuvo más cancelaciones?


### 6) Reporte final

Escriban un bloque de código que imprima un reporte con los resultados de los cuatro puntos anteriores, en formato legible. No es necesario que sea elaborado — lo importante es que quede claro qué encontraron.

In [10]:
import sqlite3
import pandas as pd

# Conexión a la base de datos
conexion = sqlite3.connect("medicare.db")

# ==========================================
# PREGUNTA 1: Turnos realizados por cada médico
# ==========================================
p1_query = """
    SELECT m.nombre AS medico, m.especialidad, COUNT(t.id) AS total_turnos
    FROM medicos m
    JOIN turnos t ON m.id = t.medico_id
    WHERE LOWER(TRIM(t.estado)) = 'realizado'
    GROUP BY m.id, m.nombre, m.especialidad
    ORDER BY total_turnos DESC;
"""
df_p1 = pd.read_sql_query(p1_query, conexion)

# ==========================================
# PREGUNTA 2: Costo total facturado por especialidad (solo turnos realizados)
# ==========================================
# Nota: Como el costo original es TEXT en la BD, usamos CAST o lo manejamos desde nuestros DataFrames limpios si ya los subimos a sqlite. 
# Para mayor seguridad, unimos y calculamos directo en Pandas con df_turnos_limpio:
df_turnos_sql = pd.read_sql_query("SELECT * FROM turnos", conexion)
df_medicos_sql = pd.read_sql_query("SELECT * FROM medicos", conexion)
df_pacientes_sql = pd.read_sql_query("SELECT * FROM pacientes", conexion)

# Usamos nuestro DataFrame limpio de turnos que ya tiene el costo numérico y estado normalizado
df_turnos_temp = df_turnos_limpio.copy()
df_turnos_temp['estado'] = df_turnos_temp['estado'].str.lower().str.strip().replace('realzado', 'realizado')

df_merged_turnos = df_turnos_temp.merge(df_medicos_sql, left_on='medico_id', right_on='id')
p2_resultado = (
    df_merged_turnos[df_merged_turnos['estado'] == 'realizado']
    .groupby('especialidad')['costo']
    .sum()
    .reset_index()
    .rename(columns={'costo': 'total_facturado'})
    .sort_values(by='total_facturado', ascending=False)
)

# ==========================================
# PREGUNTA 3: Obra social con más pacientes y pacientes sin obra social
# ==========================================
df_pacientes_temp = df_pacientes_limpio.copy()
df_pacientes_temp['obra_social'] = df_pacientes_temp['obra_social'].fillna('Sin obra social')

p3_obra_social_top = df_pacientes_temp['obra_social'].value_counts().idxmax()
p3_sin_obra_social = df_pacientes_temp[df_pacientes_temp['obra_social'].str.lower().str.contains('sin obra social|particular|ninguna', na=False)].shape[0]
# O si prefieres el conteo exacto de los que llenamos como 'Sin obra social':
conteo_obras = df_pacientes_temp['obra_social'].value_counts()

# ==========================================
# PREGUNTA 4: Turnos cancelados o ausentes y médico con más cancelaciones
# ==========================================
df_merged_cancelados = df_turnos_temp.merge(df_medicos_sql, left_on='medico_id', right_on='id')
turnos_no_realizados = df_merged_cancelados[df_merged_cancelados['estado'].isin(['cancelado', 'ausente'])]
cantidad_no_realizados = len(turnos_no_realizados)

medico_mas_cancelaciones = (
    df_merged_cancelados[df_merged_cancelados['estado'] == 'cancelado']
    ['nombre']
    .value_counts()
    .idxmax() if not df_merged_cancelados[df_merged_cancelados['estado'] == 'cancelado'].empty else "Ninguno"
)

# ==========================================
# REPORTE FINAL (Paso 6)
# ==========================================
print("=" * 60)
print("             REPORTE CLÍNICA MEDICARE             ")
print("=" * 60)

print("\n1. TURNOS REALIZADOS POR MÉDICO:")
print(df_p1.to_string(index=False))

print("\n" + "-" * 60)
print("2. COSTO TOTAL FACTURADO POR ESPECIALIDAD (Turnos Realizados):")
print(p2_resultado.to_string(index=False))

print("\n" + "-" * 60)
print("3. ANÁLISIS DE OBRA SOCIAL:")
print(f"   - Obra social con más pacientes registrados: {p3_obra_social_top}")
print(f"   - Pacientes sin obra social / cobertura: {df_pacientes_temp['obra_social'].value_counts().get('Sin obra social', 0)}")

print("\n" + "-" * 60)
print("4. AUSENTISMO Y CANCELACIONES:")
print(f"   - Total de turnos cancelados o ausentes: {cantidad_no_realizados}")
print(f"   - Médico con mayor cantidad de cancelaciones: {medico_mas_cancelaciones}")
print("=" * 60)

             REPORTE CLÍNICA MEDICARE             

1. TURNOS REALIZADOS POR MÉDICO:
             medico   especialidad  total_turnos
  Dra. Beatriz Sosa Clínica Médica            10
Dra. Carmen Aguirre    Cardiología            10
Dra. Carmen Aguirre  Traumatología            10
 Dr. Sebastián Ríos   Dermatología             9
 Dr. Sebastián Ríos  Traumatología             7
 Dra. Laura Giménez      Pediatría             7
 Dra. Gabriela Luna    Psiquiatría             7
   Dr. Nicolás Vera      Pediatría             6
 Dra. Andrea Blanco    Ginecología             6
 Dr. Ezequiel Bravo   Dermatología             6
   Dr. Pablo Méndez Clínica Médica             5
  Dra. Beatriz Sosa Clínica Médica             5
   Dr. Nicolás Vera Endocrinología             5
   Dr. Martín Cufré      Pediatría             5
   Dr. Pablo Méndez    Cardiología             4
  Dra. Silvina Vega    Psiquiatría             4
  Dra. Verónica Paz  Traumatología             3
  Dra. Beatriz Sosa     Neurologí

→ [Ir a la solución](./solucion.md)